In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())

2.2.2


In [2]:
import xarray as xr
import pandas as pd

# --------------------------------------------------
# 1. Load extremes file
# --------------------------------------------------
extremes_file = "txx_day0_top5_extremes.nc"
extremes_ds = xr.open_dataset(extremes_file)

# --------------------------------------------------
# 2. Extract dates
# --------------------------------------------------
tasmax_dates = pd.to_datetime(extremes_ds["tasmax_date"].values)
tbound_dates = pd.to_datetime(extremes_ds["t_bound_date"].values)

In [3]:
print("TASMAX Day-0 extreme dates:")
for i, d in enumerate(tasmax_dates, start=1):
    print(f"Rank {i}: {d.date()}")

print("\nT_BOUND Day-0 extreme dates:")
for i, d in enumerate(tbound_dates, start=1):
    print(f"Rank {i}: {d.date()}")


TASMAX Day-0 extreme dates:
Rank 1: 2019-07-25
Rank 2: 2022-07-19
Rank 3: 2003-08-12
Rank 4: 2015-07-01
Rank 5: 2012-08-18

T_BOUND Day-0 extreme dates:
Rank 1: 1947-07-28
Rank 2: 1961-08-29
Rank 3: 1950-06-29
Rank 4: 1969-07-28
Rank 5: 1988-08-18


In [4]:
import os
import xarray as xr
import pandas as pd
from cdo import Cdo

# =========================
# Paths
# =========================
data_path_pl = "/pool/data/ERA5/E5/pl/an/1D/"
data_path_sf = "/pool/data/ERA5/E5/sf/an/1D/"
scratch_path = "/scratch/u/u301827/paris/maps/"
final_path   = "/work/uc1275/u301827/02_MSE/paris/maps/"

os.makedirs(scratch_path, exist_ok=True)
os.makedirs(final_path,   exist_ok=True)

# =========================
# Climatology window
# =========================
def get_climatology_years(year):
    if year < 1971:
        return year + 1, year + 20
    else:
        return year - 20 , year - 1


# =========================
# Main function
# =========================
def process_era5_map_total_and_anomaly(date, var_num, var):

    cdo = Cdo()

    year, month, day = date.year, date.month, date.day
    clim_start, clim_end = get_climatology_years(year)

    date_str = date.strftime("%Y-%m-%d")
    time_window = pd.date_range(
        date - pd.Timedelta(days=3),
        date + pd.Timedelta(days=3),
        freq="D"
    )

    var_scratch = os.path.join(scratch_path, var)
    var_final   = os.path.join(final_path, var)
    os.makedirs(var_scratch, exist_ok=True)
    os.makedirs(var_final,   exist_ok=True)

    out_file = os.path.join(
        var_final,
        f"{var}_total_and_anomaly_{date_str}_pm3d.nc"
    )

    # ---------------------------------------------------------
    # Variable & level handling
    # ---------------------------------------------------------
    if var in ["t", "z"]:
        data_path = data_path_pl
        level_sel = "50000"
        prefix = "pl"
    elif var in ["sp", "blh"]:
        data_path = data_path_sf
        level_sel = None
        prefix = "sf"
    else:
        raise ValueError(f"Unsupported variable: {var}")

    # ---------------------------------------------------------
    # Determine required months
    # ---------------------------------------------------------
    months_needed = sorted(
        {(t.year, t.month) for t in time_window}
    )

    da_event_list = []

    for y, m in months_needed:

        event_file = (
            f"{data_path}{var_num}/"
            f"E5{prefix}00_1D_{y}-{m:02d}_{var_num}.grb"
        )

        if not os.path.exists(event_file):
            continue

        event_nc = os.path.join(
            var_scratch, f"event_{var}_{y}_{m:02d}.nc"
        )

        cdo.setgridtype(
            "regular",
            input=event_file,
            output=event_nc,
            options="-f nc --eccodes"
        )

        if level_sel:
            sel_event = event_nc.replace(".nc", "_sel.nc")
            cdo.sellevel(level_sel, input=event_nc, output=sel_event)
            os.remove(event_nc)
            event_nc = sel_event

        ds = xr.open_dataset(event_nc)
        varname = list(ds.data_vars)[0]
        ds["time"] = pd.to_datetime(ds["time"].values).normalize()

        da_event_list.append(ds[varname])

    if not da_event_list:
        print(f"No ERA5 data found for {date_str}")
        return None

    da_event = xr.concat(da_event_list, dim="time")
    da_event = da_event.sel(time=time_window)

    # ---------------------------------------------------------
    # Load climatology (same calendar day only)
    # ---------------------------------------------------------
    clim_list = []

    for y in range(clim_start, clim_end + 1):

        clim_file = (
            f"{data_path}{var_num}/"
            f"E5{prefix}00_1D_{y}-{month:02d}_{var_num}.grb"
        )

        if not os.path.exists(clim_file):
            continue

        clim_nc = os.path.join(var_scratch, f"clim_{var}_{y}.nc")

        cdo.setgridtype(
            "regular",
            input=clim_file,
            output=clim_nc,
            options="-f nc --eccodes"
        )

        if level_sel:
            clim_sel = clim_nc.replace(".nc", "_sel.nc")
            cdo.sellevel(level_sel, input=clim_nc, output=clim_sel)
            os.remove(clim_nc)
            clim_nc = clim_sel

        ds = xr.open_dataset(clim_nc)
        ds["time"] = pd.to_datetime(ds["time"].values).normalize()

        try:
            clim_list.append(
                ds[varname].sel(time=f"{y}-{month:02d}-{day:02d}")
            )
        except KeyError:
            pass

    clim_mean = xr.concat(clim_list, dim="time").mean("time")

    # ---------------------------------------------------------
    # Build output Dataset
    # ---------------------------------------------------------
    ds_out = xr.Dataset(
        {
            var: da_event,
            f"{var}_anom": da_event - clim_mean,
        }
    )

    ds_out.to_netcdf(out_file)

    return out_file



In [5]:
for d in tasmax_dates: print(d)

2019-07-25 00:00:00
2022-07-19 00:00:00
2003-08-12 00:00:00
2015-07-01 00:00:00
2012-08-18 00:00:00


In [ ]:
from joblib import Parallel, delayed

# list of variables to process
var_list = [(130, "t"), (129, "z"), (134, "sp"), (159, "blh")]

#process_era5_map_total_and_anomaly(tasmax_dates[3], 159, "blh")
Parallel(n_jobs=30, verbose=10)(
    delayed(process_era5_map_total_and_anomaly)(d, var_num, var)
    for d in tasmax_dates
    for var_num, var in var_list
)

[Parallel(n_jobs=30)]: Using backend LokyBackend with 30 concurrent workers.
[Parallel(n_jobs=30)]: Done   3 out of  20 | elapsed:   23.1s remaining:  2.2min
[Parallel(n_jobs=30)]: Done   6 out of  20 | elapsed:   24.0s remaining:   56.1s
[Parallel(n_jobs=30)]: Done   9 out of  20 | elapsed:   25.2s remaining:   30.8s


In [ ]:
from joblib import Parallel, delayed

# list of variables to process
var_list = [(130, "t"), (129, "z"), (134, "sp"), (159, "blh")]

Parallel(n_jobs=30, verbose=10)(
    delayed(process_era5_map_total_and_anomaly)(d, var_num, var)
    for d in tbound_dates
    for var_num, var in var_list
)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

# -----------------------------
# Paths
# -----------------------------
t_path   = "/work/uc1275/u301827/02_MSE/paris/maps/t/"
z_path   = "/work/uc1275/u301827/02_MSE/paris/maps/z/"
blh_path = "/work/uc1275/u301827/02_MSE/paris/maps/blh/"

save_path = "/work/uc1275/u301827/00_codephd/mrt_cmip6/DKRZ/plots_top5tbound/"
os.makedirs(save_path, exist_ok=True)

# -----------------------------
# Dates
# -----------------------------
tasmax_dates = tbound_dates  # list of datetime objects

# -----------------------------
# Region
# -----------------------------
lat_slice = slice(65, 35)   # lat descending
lon_slice = slice(-20, 20)

# -----------------------------
# Loop over events
# -----------------------------
for i, date in enumerate(tasmax_dates, start=1):

    date_str = date.strftime("%Y-%m-%d")
    print(f"Processing event {i}: {date_str}")

    t_file   = os.path.join(t_path,   f"t_total_and_anomaly_{date_str}_pm3d.nc")
    z_file   = os.path.join(z_path,   f"z_total_and_anomaly_{date_str}_pm3d.nc")
    blh_file = os.path.join(blh_path, f"blh_total_and_anomaly_{date_str}_pm3d.nc")

    if not all(os.path.exists(f) for f in [t_file, z_file, blh_file]):
        print(f"Missing file(s) for {date_str}, skipping...")
        continue

    # -----------------------------
    # Open datasets
    # -----------------------------
    ds_t   = xr.open_dataset(t_file)
    ds_z   = xr.open_dataset(z_file)
    ds_blh = xr.open_dataset(blh_file)

    t_anom = ds_t["t_anom"].isel(plev=0)
    z500   = ds_z["z"].isel(plev=0)
    blh_an = ds_blh["blh_anom"]

    # -----------------------------
    # Fix longitudes
    # -----------------------------
    def fix_lon(da):
        return da.assign_coords(
            lon=((da.lon + 180) % 360) - 180
        ).sortby("lon")

    t_anom = fix_lon(t_anom)
    z500   = fix_lon(z500)
    blh_an = fix_lon(blh_an)

    # -----------------------------
    # Select Western Europe
    # -----------------------------
    t_we   = t_anom.sel(lat=lat_slice, lon=lon_slice)
    z_we   = z500.sel(lat=lat_slice, lon=lon_slice)
    blh_we = blh_an.sel(lat=lat_slice, lon=lon_slice)

    times = t_we.time.values
    ntime = len(times)

    # -----------------------------
    # Figure layout
    # -----------------------------
    fig, axes = plt.subplots(
        nrows=2,
        ncols=ntime,
        figsize=(3.2 * ntime, 7),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True
    )

    if ntime == 1:
        axes = axes.reshape(2, 1)

    # -----------------------------
    # Loop over time slices
    # -----------------------------
    for j, t in enumerate(times):

        date_label = xr.DataArray(t).dt.strftime("%Y-%m-%d").item()

        # ---- Row 1: T anomaly + Z500 ----
        ax = axes[0, j]

        if j == ntime - 1:
            t_we.sel(time=t).plot.contourf(
                ax=ax,
                cmap="bwr",
                vmin=-6, vmax=6,
                levels=13,
                add_colorbar=True,
                cbar_kwargs={"label": "T anomaly (K)"}
            )
        else:
            t_we.sel(time=t).plot.contourf(
                ax=ax,
                cmap="bwr",
                vmin=-6, vmax=6,
                levels=13,
                add_colorbar=False
            )

        z_we.sel(time=t).plot.contour(
            ax=ax,
            colors="k",
            linewidths=1,
            levels=15
        )

        ax.coastlines(resolution="10m")
        ax.add_feature(cfeature.BORDERS, linestyle=":")
        ax.set_title(date_label)

        # ---- Row 2: BLH anomaly ----
        ax = axes[1, j]

        if j == ntime - 1:
            blh_we.sel(time=t).plot.contourf(
                ax=ax,
                cmap="BrBG",
                levels=13,
                add_colorbar=True,
                cbar_kwargs={"label": "BLH anomaly (m)"}
            )
        else:
            blh_we.sel(time=t).plot.contourf(
                ax=ax,
                cmap="BrBG",
                levels=13,
                add_colorbar=False
            )

        ax.coastlines(resolution="10m")
        ax.add_feature(cfeature.BORDERS, linestyle=":")

    axes[0, 0].set_ylabel("T500 anom + Z500")
    axes[1, 0].set_ylabel("BLH anomaly")

    fig.suptitle(
        f"ERA5 ±3 day evolution around {date_str}",
        fontsize=14
    )

    # -----------------------------
    # Save
    # -----------------------------
    out_file = os.path.join(
        save_path, f"t500_z500_blh_pm3d_{date_str}.png"
    )

    plt.savefig(out_file, dpi=150, bbox_inches="tight")
    plt.close(fig)


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

# -----------------------------
# Paths
# -----------------------------
t_path = "/work/uc1275/u301827/02_MSE/paris/maps/t/"
z_path = "/work/uc1275/u301827/02_MSE/paris/maps/z/"
save_path = "/work/uc1275/u301827/00_codephd/mrt_cmip6/DKRZ/plots_top5tbound/"
os.makedirs(save_path, exist_ok=True)

# -----------------------------
# Dates (example)
# -----------------------------
tasmax_dates = tbound_dates  # list of datetime objects

# -----------------------------
# Region
# -----------------------------
lat_slice = slice(65, 35)  # lat descending
lon_slice = slice(-20, 20)

# -----------------------------
# Loop over dates
# -----------------------------
for i, date in enumerate(tasmax_dates, start=1):
    date_str = date.strftime("%Y-%m-%d")
    print(f"Processing date {i}: {date_str}")

    t_file = os.path.join(t_path, f"t_total_and_anomaly_{date_str}.nc")
    z_file = os.path.join(z_path, f"z_total_and_anomaly_{date_str}.nc")

    if not os.path.exists(t_file) or not os.path.exists(z_file):
        print(f"Missing file for {date_str}, skipping...")
        continue

    # -----------------------------
    # Open datasets
    # -----------------------------
    ds_t = xr.open_dataset(t_file)
    ds_z = xr.open_dataset(z_file)

    t_anom = ds_t["t_anom"].isel(plev=0)
    z500   = ds_z["z"].isel(plev=0)

    # -----------------------------
    # Fix longitudes and sort
    # -----------------------------
    t_anom = t_anom.assign_coords(lon=((t_anom.lon + 180) % 360) - 180).sortby("lon")
    z500   = z500.assign_coords(lon=((z500.lon + 180) % 360) - 180).sortby("lon")

    # -----------------------------
    # Select Western Europe
    # -----------------------------
    t_we = t_anom.sel(lat=lat_slice, lon=lon_slice)
    z_we = z500.sel(lat=lat_slice, lon=lon_slice)

    # -----------------------------
    # Plot
    # -----------------------------
    fig, ax = plt.subplots(
        figsize=(10,7),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )

    # Temperature anomaly (colors)
    t_we.plot.contourf(
        ax=ax,
        cmap="bwr",
        vmin=-6, vmax=6,
        add_colorbar=True,
        levels=13
    )

    # Z500 contours (black lines)
    z_we.plot.contour(
        ax=ax,
        colors="k",
        linewidths=1,
        levels=15,
        add_labels=True
    )

    # Coastlines & borders
    ax.coastlines(resolution="10m")
    ax.add_feature(cfeature.BORDERS, linestyle=":", edgecolor="black")
    ax.set_title(f"500 hPa Temperature Anomaly + Z500 Contours\nDate: {date_str}")

    # Save figure
    out_file = os.path.join(save_path, f"t500_z500_{date_str}.png")
    plt.savefig(out_file, dpi=150, bbox_inches="tight")
    plt.close(fig)


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# -----------------------------
# Constants
# -----------------------------
R   = 287.0       # J/kg/K, dry air
cp  = 1004.0      # J/kg/K, dry air
p_ref = 500e2     # 500 hPa in Pa

# -----------------------------
# File paths
# -----------------------------
t_file  = "/work/uc1275/u301827/02_MSE/paris/maps/t/t_total_and_anomaly_2019-07-25.nc"
sp_file = "/work/uc1275/u301827/02_MSE/paris/maps/sp/sp_total_and_anomaly_2019-07-25.nc"

# -----------------------------
# Open datasets
# -----------------------------
ds_t  = xr.open_dataset(t_file)
ds_sp = xr.open_dataset(sp_file)

# Temperature at 500 hPa
T500 = ds_t["t"].isel(plev=0)  # K
# Surface pressure / SLP
ps   = ds_sp["sp"]              # Pa

# -----------------------------
# Compute potential temperature at 500 hPa
# θ500 = T500 * (ps / p_ref)^(R/cp)
# -----------------------------
theta500 = T500 * (ps / p_ref)**(R / cp) - 273.15
ds_t["theta500"] = theta500

# -----------------------------
# Fix longitudes
# -----------------------------
theta500 = theta500.assign_coords(lon=((theta500.lon + 180)  % 360) - 180).sortby("lon")
ps = ps.assign_coords(lon=((ps.lon + 180)  % 360) - 180).sortby("lon")

# -----------------------------
# Western Europe subset
# -----------------------------
lat_slice = slice(65, 35)
lon_slice = slice(-20, 20)

theta_we = theta500.sel(lat=lat_slice, lon=lon_slice)
slp_we   = ps.sel(lat=lat_slice, lon=lon_slice)

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(
    figsize=(10,7),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

# Potential temperature as color map
pcm = theta_we.plot.contourf(
    ax=ax,
    cmap="coolwarm",
    levels=15,
    add_colorbar=True
)

# SLP contours
contours = slp_we.plot.contour(
    ax=ax,
    colors="k",
    linewidths=1,
    levels=15,
    add_labels=True
)

# Coastlines and borders
ax.coastlines(resolution="10m")
ax.add_feature(cfeature.BORDERS, linestyle=":", edgecolor="black")
ax.set_title("Dry_bound (colours) + SLP (contours)")

plt.show()
